# **<div align="center">TARGET DEFINITION</div>**

This notebook builds the prediction targets for volatile cryptocurrencies and stablecoins.
- **Volatile cryptocurrencies**: three-class future-return classification.
- **Stable cryotocurrencies**: binary classification of any future deviation from the USD peg.
Target columns represent future outcomes. They must not be used as model input features.

In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## **1. Configuration**

In [19]:
import numpy as np
import pandas as pd

# Forecast horizons, expressed in days.
HORIZONS = {
    "short": 7,
    "medium": 21,
    "long": 50
}

# Cryptocurrencies included in the experiment.
CRYPTO_NAMES = [
    "bitcoin",
    "ethereum",
    "tether",
    "binancecoin",
    "usd-coin"
]

# Asset groups with different target definitions.
VOLATILE_CRYPTOS = [
    "bitcoin",
    "ethereum",
    "binancecoin"
]

STABLE_CRYPTOS = [
    "tether",
    "usd-coin"
]

# Data and chronological split configuration.
DATA_PATH = "../data/processed/"
TRAIN_RATIO = 0.8

# Future-return thresholds for volatile cryptocurrencies.
RETURN_THRESHOLDS = {
    "short": 0.03,
    "medium": 0.05,
    "long": 0.08
}

# Maximum absolute deviation from the $1 peg, expressed as a percentage.
PEG_TOLERANCE = 0.02

## **2. Target classification functions**

In [20]:

def classify_return(return_value, threshold):
    '''
    Classify future returns into downward, sideways, or upward trends.
    '''
    if pd.isna(return_value):
        return np.nan
    elif return_value <= -threshold:
        return -1
    elif return_value < threshold:
        return 0
    else:
        return 1


## **3. Import dataset**

In [21]:
# Load each asset and use the date as the sorted index.
dfs = {}

for crypto in CRYPTO_NAMES:
    path = f"{DATA_PATH}{crypto}.csv"
    df = pd.read_csv(path)

    df["date"] = pd.to_datetime(df["date"])
    df = df.set_index("date", drop=False).sort_index()

    dfs[crypto] = df

    print(
        f"{crypto}: {len(df)} observations | "
        f"{df.index.min().date()} -> {df.index.max().date()}"
    )

bitcoin: 281 observations | 2025-02-05 -> 2025-11-12
ethereum: 241 observations | 2025-02-05 -> 2025-10-03
tether: 281 observations | 2025-02-05 -> 2025-11-12
binancecoin: 240 observations | 2025-02-05 -> 2025-10-02
usd-coin: 280 observations | 2025-02-06 -> 2025-11-12


## **4. Calculate future prices and returns**

In [22]:
# Calculate the price and return at the end of each forecast horizon.
for crypto, df in dfs.items():
    for horizon_name, horizon in HORIZONS.items():
        future_price = df["price_usd"].shift(-horizon)

        df[f"target_price_{horizon_name}"] = future_price
        df[f"future_return_{horizon_name}"] = (
            future_price / df["price_usd"] - 1
        )

## **5. Chronological train/test split**

In [23]:
# Split each asset chronologically; do not shuffle time-series observations.
train_dfs = {}
test_dfs = {}

for crypto, df in dfs.items():
    split_idx = int(len(df) * TRAIN_RATIO)

    train_dfs[crypto] = df.iloc[:split_idx].copy()
    test_dfs[crypto] = df.iloc[split_idx:].copy()

    print(
        f"{crypto}: train={len(train_dfs[crypto])}, "
        f"test={len(test_dfs[crypto])}, "
        f"train_end={train_dfs[crypto].index.max().date()}, "
        f"test_start={test_dfs[crypto].index.min().date()}"
    )

bitcoin: train=224, test=57, train_end=2025-09-16, test_start=2025-09-17
ethereum: train=192, test=49, train_end=2025-08-15, test_start=2025-08-16
tether: train=224, test=57, train_end=2025-09-16, test_start=2025-09-17
binancecoin: train=192, test=48, train_end=2025-08-15, test_start=2025-08-16
usd-coin: train=224, test=56, train_end=2025-09-17, test_start=2025-09-18


## **6. Remove training rows whose target horizon overlaps the test period**

In [24]:
# Keep only training rows whose longest target horizon ends before the test period.
train_target_dfs = {}

for crypto, train_df in train_dfs.items():
    valid_rows = pd.Series(True, index=train_df.index)
    test_start = test_dfs[crypto].index.min()

    for horizon in HORIZONS.values():
        future_date = train_df.index + pd.to_timedelta(horizon, unit="D")
        valid_rows &= future_date < test_start

    train_target_dfs[crypto] = train_df.loc[valid_rows].copy()

    print(
        f"{crypto}: {len(train_df)} original training rows, "
        f"{len(train_target_dfs[crypto])} valid target rows, "
        f"{len(train_df) - len(train_target_dfs[crypto])} excluded"
    )

bitcoin: 224 original training rows, 174 valid target rows, 50 excluded
ethereum: 192 original training rows, 142 valid target rows, 50 excluded
tether: 224 original training rows, 174 valid target rows, 50 excluded
binancecoin: 192 original training rows, 142 valid target rows, 50 excluded
usd-coin: 224 original training rows, 174 valid target rows, 50 excluded


## **7. Check daily date continuity**

In [25]:
# Check that each dataset contains consecutive daily observations.
for crypto, df in dfs.items():
    date_diffs = df.index.to_series().diff().dropna()
    gaps = date_diffs[date_diffs != pd.Timedelta(days=1)]

    print(f"{crypto}: {len(gaps)} gaps in daily observations")

bitcoin: 0 gaps in daily observations
ethereum: 0 gaps in daily observations
tether: 0 gaps in daily observations
binancecoin: 0 gaps in daily observations
usd-coin: 0 gaps in daily observations


## **8. Volatile cryptocurrencies: future-return targets**

The target is based on the cumulative return at the end of each forecast horizon.
| Horizon | Downward | Sideways | Upward |
| --- | --- | --- | --- |
| Short (7 days) | ≤ −3% | between −3% and +3% | ≥ +3% |
| Medium (21 days)	| ≤ −5%	| between −5% and +5%	| ≥ +5% |
| Long (50 days)	| ≤ −8%	| between −8% and +8% | ≥ +8% |


Labels: -1 = downward, 0 = sideways, 1 = upward.

In [26]:
# Create return-based target labels for volatile cryptocurrencies.
for crypto in VOLATILE_CRYPTOS:
    df = train_target_dfs[crypto]

    for horizon_name, threshold in RETURN_THRESHOLDS.items():
        return_col = f"future_return_{horizon_name}"
        target_col = f"trend_return_3_{horizon_name}"

        df[target_col] = df[return_col].apply(
            lambda value: classify_return(value, threshold)
        )

    train_target_dfs[crypto] = df

In [27]:
# Summarize the class counts and percentages for each asset and horizon.
volatile_target_summary = []

for crypto in VOLATILE_CRYPTOS:
    df = train_target_dfs[crypto]

    for horizon_name in HORIZONS:
        target_col = f"trend_return_3_{horizon_name}"
        counts = df[target_col].value_counts().reindex([-1, 0, 1], fill_value=0)
        total = counts.sum()

        for label, count in counts.items():
            volatile_target_summary.append({
                "crypto": crypto,
                "horizon": horizon_name,
                "class": label,
                "count": int(count),
                "percentage": 100 * count / total if total else np.nan
            })

volatile_target_summary_df = pd.DataFrame(volatile_target_summary)
display(volatile_target_summary_df.round(2))

,crypto,horizon,class,count,percentage
0,bitcoin,short,-1,32,18.39
1,bitcoin,short,0,99,56.90
2,bitcoin,short,1,43,24.71
3,bitcoin,medium,-1,30,17.24
4,bitcoin,medium,0,81,46.55
5,bitcoin,medium,1,63,36.21
6,bitcoin,long,-1,23,13.22
7,bitcoin,long,0,60,34.48
8,bitcoin,long,1,91,52.30
9,ethereum,short,-1,49,34.51


## **9. Stablecoins: future peg-deviation targets**

For each observation, inspect every daily price in the future horizon, excluding the current date.

The maximum absolute deviation from the USD peg is calculated as:

$$
\text{Max deviation (\%)} =
100 \times \max_{1 \leq t \leq h}
\left|P_{d+t} - 1\right|
$$


- **0** = the price remains within the selected peg tolerance throughout the horizon.
- **1** = at least one future observation exceeds the tolerance.
The selected tolerance is 0.02% from the $1 peg. This is an experimental threshold for this dataset, not a universal definition of a depeg.

In [28]:
# Create the maximum future peg deviation and binary target for each stablecoin.
for crypto in STABLE_CRYPTOS:
    df = train_target_dfs[crypto]
    full_df = dfs[crypto]

    for horizon_name, horizon in HORIZONS.items():
        max_deviations = []

        for date in df.index:
            future_dates = pd.date_range(
                start=date + pd.Timedelta(days=1),
                periods=horizon,
                freq="D"
            )

            future_prices = full_df.reindex(future_dates)["price_usd"]

            # Require a complete future window before assigning a target.
            if future_prices.notna().sum() != horizon:
                max_deviation = np.nan
            else:
                max_deviation = (future_prices.sub(1).abs().max() * 100)

            max_deviations.append(max_deviation)

        deviation_col = f"max_peg_deviation_{horizon_name}"
        target_col = f"target_peg_{horizon_name}"

        df[deviation_col] = max_deviations
        df[target_col] = np.where(
            df[deviation_col].notna(),
            (df[deviation_col] > PEG_TOLERANCE).astype(int),
            np.nan
        )

    train_target_dfs[crypto] = df

In [29]:
# Summarize stablecoin target counts for each horizon.
stablecoin_target_summary = []

for crypto in STABLE_CRYPTOS:
    df = train_target_dfs[crypto]

    for horizon_name in HORIZONS:
        target_col = f"target_peg_{horizon_name}"
        counts = df[target_col].value_counts().reindex([0, 1], fill_value=0)
        total = counts.sum()

        for label, count in counts.items():
            stablecoin_target_summary.append({
                "crypto": crypto,
                "horizon": horizon_name,
                "class": int(label),
                "class_name": "Maintains peg" if label == 0 else "Peg deviation",
                "count": int(count),
                "percentage": 100 * count / total if total else np.nan
            })

stablecoin_target_summary_df = pd.DataFrame(stablecoin_target_summary)
display(stablecoin_target_summary_df.round(2))

,crypto,horizon,class,class_name,count,percentage
0,tether,short,0,Maintains peg,104,59.77
1,tether,short,1,Peg deviation,70,40.23
2,tether,medium,0,Maintains peg,67,38.51
3,tether,medium,1,Peg deviation,107,61.49
4,tether,long,0,Maintains peg,22,12.64
5,tether,long,1,Peg deviation,152,87.36
6,usd-coin,short,0,Maintains peg,106,60.92
7,usd-coin,short,1,Peg deviation,68,39.08
8,usd-coin,medium,0,Maintains peg,62,35.63
9,usd-coin,medium,1,Peg deviation,112,64.37


## **10. Final target columns**

**Volatile cryptocurrencies**
- trend_return_3_short
- trend_return_3_medium
- trend_return_3_long

**Stablecoins**
- max_peg_deviation_short, max_peg_deviation_medium, max_peg_deviation_long
- target_peg_short, target_peg_medium, target_peg_long
The resulting train_target_dfs contains the valid training observations and their target columns. Future-derived columns are labels only and must be excluded from model features.